# RBAC Audit Notebook

Portable role-based access control audit for a new Snowflake environment. Run top to
bottom. It surfaces:

1. **Role hierarchy** — how roles inherit from each other, and how deep/tangled it is
2. **Powerful role membership** — who holds ACCOUNTADMIN, SECURITYADMIN, SYSADMIN, and other admin-level roles
3. **Broad/high-risk grants** — privileges like `ALL PRIVILEGES`, account-level `OWNERSHIP`, or grants on the whole database/schema rather than specific objects
4. **Orphaned roles** — roles with no users assigned and/or no grants, that are just clutter (or worse, forgotten backdoors)
5. **Users with excessive role counts** — a proxy for access sprawl and for accounts that are hard to reason about
6. **Individual (non-role) object ownership** — objects owned directly by a user rather than a role, which breaks the RBAC model and creates single points of failure
7. **A rolled-up per-role and per-user risk summary**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database (or `ACCOUNTADMIN`), so you can
  query `SNOWFLAKE.ACCOUNT_USAGE.*`.
- `ACCOUNT_USAGE` views here (`GRANTS_TO_ROLES`, `GRANTS_TO_USERS`, `ROLES`, `USERS`) reflect
  the *current* state of grants (with history via `deleted_on`), not a point-in-time snapshot —
  this notebook is safe to re-run at any point in the engagement to see how the picture changes.
- Nothing here is destructive — every cell is a `SELECT`/`SHOW`. Nothing writes to the account.

### A note on FUTURE GRANTS
`ACCOUNT_USAGE` doesn't expose future grants (grants that auto-apply to objects not created
yet) as a single account-wide view — they have to be checked per database/schema with
`SHOW FUTURE GRANTS IN DATABASE <name>` / `SHOW FUTURE GRANTS IN SCHEMA <name>`. That's called
out as a manual follow-up near the end of this notebook rather than automated here, since the
practical approach is to check it on the specific databases/schemas this audit flags as
high-risk, not sweep the whole account blindly.


In [ ]:
-- ============================================================
-- PARAMETERS — adjust once per environment.
-- ============================================================
SET admin_roles = 'ACCOUNTADMIN,SECURITYADMIN,SYSADMIN,ORGADMIN';
SET excessive_role_count = 10;      -- flag users holding more than this many roles
SET stale_login_days = 90;          -- flag users/roles inactive longer than this

SELECT $admin_roles AS admin_roles,
       $excessive_role_count AS excessive_role_count,
       $stale_login_days AS stale_login_days;


## 1. Role hierarchy overview

Roles granted to other roles (`GRANTED_ON = 'ROLE'` in `GRANTS_TO_ROLES`) form the inheritance
tree. A healthy hierarchy is shallow and intentional (functional roles → access roles →
admin roles); a tangled one is a sign access has grown organically without a model behind it.


In [ ]:
-- Role -> role grants (the inheritance edges)
SELECT
    grantee_name   AS parent_role,
    name           AS child_role_granted_into_parent,
    granted_by,
    created_on
FROM snowflake.account_usage.grants_to_roles
WHERE granted_on = 'ROLE'
  AND deleted_on IS NULL
ORDER BY parent_role, child_role_granted_into_parent;


In [ ]:
-- Roles with the most direct children (hierarchy hotspots) and roles with no children/no parents (isolated)
WITH edges AS (
    SELECT grantee_name AS parent_role, name AS child_role
    FROM snowflake.account_usage.grants_to_roles
    WHERE granted_on = 'ROLE' AND deleted_on IS NULL
),
parent_counts AS (
    SELECT parent_role, COUNT(*) AS direct_children
    FROM edges
    GROUP BY parent_role
),
child_counts AS (
    SELECT child_role, COUNT(*) AS direct_parents
    FROM edges
    GROUP BY child_role
)
SELECT
    r.name AS role_name,
    r.owner,
    r.assigned_to_users,
    COALESCE(pc.direct_children, 0) AS direct_children_roles,
    COALESCE(cc.direct_parents, 0)  AS direct_parent_roles,
    r.created_on
FROM snowflake.account_usage.roles r
LEFT JOIN parent_counts pc ON r.name = pc.parent_role
LEFT JOIN child_counts cc ON r.name = cc.child_role
WHERE r.deleted_on IS NULL
ORDER BY direct_children_roles DESC, direct_parent_roles DESC;


## 2. Powerful role membership

Who can do `ACCOUNTADMIN`-level (or other admin-role) things is the single highest-risk
question in an RBAC audit — this should be a short, deliberate list, not something that has
grown by accretion.


In [ ]:
-- Users holding admin-level roles, directly
SELECT
    gu.role            AS admin_role,
    gu.grantee_name    AS user_name,
    u.disabled,
    u.type             AS user_type,
    u.last_success_login,
    gu.granted_by,
    gu.created_on      AS granted_on
FROM snowflake.account_usage.grants_to_users gu
LEFT JOIN snowflake.account_usage.users u ON gu.grantee_name = u.name
WHERE gu.deleted_on IS NULL
  AND gu.role IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($admin_roles, ',')))
ORDER BY admin_role, user_name;


**Watch for:** disabled users still holding an admin role (should have been revoked, not
just disabled), service-type accounts with `ACCOUNTADMIN` (should almost always be a narrower
functional role instead), and any admin-role holder who hasn't logged in within
`stale_login_days` — a stale credential with top-level access is a bigger risk than a stale
credential with narrow access.


## 3. Broad / high-risk grants

Grants that apply at the account or database level, or that grant `ALL PRIVILEGES` /
`OWNERSHIP` broadly, are the ones most likely to be wider than actually needed. This doesn't
mean every result here is wrong — some are intentional — but each one is worth a deliberate
"yes, this should exist" decision rather than being inherited unexamined.


In [ ]:
-- Broadest grants in the account: ALL PRIVILEGES, OWNERSHIP, and anything at ACCOUNT/DATABASE scope
SELECT
    grantee_name       AS role_name,
    privilege,
    granted_on         AS object_type,
    name               AS object_name,
    table_catalog      AS database_name,
    table_schema       AS schema_name,
    grant_option,
    granted_by,
    created_on
FROM snowflake.account_usage.grants_to_roles
WHERE deleted_on IS NULL
  AND (
        privilege = 'ALL PRIVILEGES'
        OR (privilege = 'OWNERSHIP' AND granted_on IN ('DATABASE', 'SCHEMA', 'ACCOUNT'))
        OR granted_on = 'ACCOUNT'
      )
ORDER BY object_type, role_name;


In [ ]:
-- Roles with grant_option = TRUE (can re-grant the privilege to others) — a second layer of risk
SELECT
    grantee_name AS role_name,
    privilege,
    granted_on   AS object_type,
    name         AS object_name,
    table_catalog AS database_name,
    granted_by,
    created_on
FROM snowflake.account_usage.grants_to_roles
WHERE deleted_on IS NULL
  AND grant_option = TRUE
ORDER BY role_name, object_type;


## 4. Orphaned roles

Roles with no users assigned and no meaningful grants are, at best, clutter that makes the
RBAC model harder to reason about, and at worst a forgotten path that still has real
privileges attached and nobody is watching. `ROLES.ASSIGNED_TO_USERS` gives the user count
directly; the second query checks whether an "unused" role still has live grants worth
reviewing before deleting it.


In [ ]:
-- Roles with zero users assigned
SELECT
    name AS role_name,
    owner,
    created_on,
    comment
FROM snowflake.account_usage.roles
WHERE deleted_on IS NULL
  AND assigned_to_users = 0
ORDER BY created_on;


In [ ]:
-- Of those unused-by-user roles, which ones still hold real privileges worth reviewing before dropping
WITH unused_roles AS (
    SELECT name AS role_name
    FROM snowflake.account_usage.roles
    WHERE deleted_on IS NULL AND assigned_to_users = 0
)
SELECT
    g.grantee_name AS role_name,
    g.privilege,
    g.granted_on   AS object_type,
    g.name         AS object_name,
    g.table_catalog AS database_name,
    g.created_on
FROM snowflake.account_usage.grants_to_roles g
JOIN unused_roles u ON g.grantee_name = u.role_name
WHERE g.deleted_on IS NULL
ORDER BY role_name, object_type;


## 5. Users with excessive role membership

A user holding many roles is harder to reason about (what can this account actually do,
in total?) and is a common side effect of "just grant them what they ask for" access
requests piling up over time instead of being modeled through a smaller set of functional
roles.


In [ ]:
-- Users ranked by number of active role grants
SELECT
    gu.grantee_name AS user_name,
    u.disabled,
    u.type          AS user_type,
    u.last_success_login,
    COUNT(*)        AS active_role_count,
    ARRAY_AGG(gu.role) WITHIN GROUP (ORDER BY gu.role) AS roles_held
FROM snowflake.account_usage.grants_to_users gu
LEFT JOIN snowflake.account_usage.users u ON gu.grantee_name = u.name
WHERE gu.deleted_on IS NULL
GROUP BY gu.grantee_name, u.disabled, u.type, u.last_success_login
HAVING COUNT(*) > $excessive_role_count
ORDER BY active_role_count DESC;


## 6. Service account & authentication review

Flags accounts by type and authentication method, so password-based or legacy-auth service
accounts (a common audit finding) are easy to find and prioritize for migration to key-pair
or OAuth.


In [ ]:
-- Users by type and authentication method
SELECT
    name AS user_name,
    type AS user_type,
    disabled,
    has_password,
    has_rsa_public_key,
    ext_authn_duo,
    default_role,
    default_warehouse,
    last_success_login,
    days_to_expiry,
    created_on
FROM snowflake.account_usage.users
WHERE deleted_on IS NULL
ORDER BY type, user_name;


In [ ]:
-- Likely service accounts (by type, or by naming convention if TYPE isn't populated) still using passwords only
SELECT
    name AS user_name,
    type AS user_type,
    disabled,
    has_password,
    has_rsa_public_key,
    last_success_login,
    created_on
FROM snowflake.account_usage.users
WHERE deleted_on IS NULL
  AND (type IN ('SERVICE', 'LEGACY_SERVICE') OR name ILIKE ANY ('%SVC%', '%SERVICE%', '%_BOT%', '%_APP%'))
  AND has_password = TRUE
  AND has_rsa_public_key = FALSE
ORDER BY user_name;


## 7. Individual (non-role) object ownership

Snowflake best practice is that every securable object is owned by a role, not a person —
otherwise, if that person leaves or is deprovisioned, the objects they own become orphaned or
require an admin to manually reassign ownership. This checks databases, schemas, tables, and
views for ownership by something that isn't clearly a functional/system role.


In [ ]:
-- Objects owned directly by what looks like an individual user rather than a role
-- (heuristic: owner name matches a user in ACCOUNT_USAGE.USERS and not a role in ACCOUNT_USAGE.ROLES)
WITH role_names AS (
    SELECT name FROM snowflake.account_usage.roles WHERE deleted_on IS NULL
),
user_names AS (
    SELECT name FROM snowflake.account_usage.users WHERE deleted_on IS NULL
)
SELECT 'DATABASE' AS object_type, database_name AS object_name, database_owner AS owner
FROM snowflake.account_usage.databases
WHERE deleted IS NULL
  AND database_owner IN (SELECT name FROM user_names)
  AND database_owner NOT IN (SELECT name FROM role_names)

UNION ALL

SELECT 'SCHEMA', schema_name, schema_owner
FROM snowflake.account_usage.schemata
WHERE deleted IS NULL
  AND schema_owner IN (SELECT name FROM user_names)
  AND schema_owner NOT IN (SELECT name FROM role_names)

UNION ALL

SELECT 'TABLE', table_catalog || '.' || table_schema || '.' || table_name, table_owner
FROM snowflake.account_usage.tables
WHERE deleted IS NULL
  AND table_owner IN (SELECT name FROM user_names)
  AND table_owner NOT IN (SELECT name FROM role_names)

ORDER BY object_type, object_name;


## 8. Rolled-up recommendation summary


In [ ]:
-- Per-role risk summary
WITH admin_membership AS (
    SELECT grantee_name AS user_name, role
    FROM snowflake.account_usage.grants_to_users
    WHERE deleted_on IS NULL
      AND role IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($admin_roles, ',')))
),
broad_grant_roles AS (
    SELECT DISTINCT grantee_name AS role_name
    FROM snowflake.account_usage.grants_to_roles
    WHERE deleted_on IS NULL
      AND (privilege = 'ALL PRIVILEGES' OR (privilege = 'OWNERSHIP' AND granted_on IN ('DATABASE','SCHEMA','ACCOUNT')) OR granted_on = 'ACCOUNT')
),
unused_roles AS (
    SELECT name AS role_name FROM snowflake.account_usage.roles WHERE deleted_on IS NULL AND assigned_to_users = 0
)
SELECT
    r.name AS role_name,
    r.assigned_to_users,
    r.owner,
    IFF(r.name IN (SELECT role FROM admin_membership), 'YES', 'no') AS is_admin_role,
    IFF(r.name IN (SELECT role_name FROM broad_grant_roles), 'YES', 'no') AS has_broad_grant,
    IFF(r.name IN (SELECT role_name FROM unused_roles), 'YES', 'no') AS zero_users_assigned,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        IFF(r.name IN (SELECT role FROM admin_membership), 'REVIEW ADMIN MEMBERSHIP', NULL),
        IFF(r.name IN (SELECT role_name FROM broad_grant_roles), 'NARROW SCOPE OF GRANTS', NULL),
        IFF(r.name IN (SELECT role_name FROM unused_roles), 'CANDIDATE FOR CLEANUP/REMOVAL', NULL)
    ), '; ') AS recommendations
FROM snowflake.account_usage.roles r
WHERE r.deleted_on IS NULL
ORDER BY (is_admin_role = 'YES') DESC, (has_broad_grant = 'YES') DESC, r.name;


### Notes, caveats, and next steps

- **This audits current state, not history.** For "who had access to X on this date" (useful
  for an incident review or compliance question), you'd query the same views with the
  `created_on`/`deleted_on` columns to reconstruct a point-in-time picture, rather than the
  current-state filters (`deleted_on IS NULL`) used throughout this notebook.
- **Future grants need a per-database/schema check.** Once Section 3 and Section 8 have
  flagged the highest-risk roles and databases, run `SHOW FUTURE GRANTS IN DATABASE <name>;`
  / `SHOW FUTURE GRANTS IN SCHEMA <name>;` against those specifically.
- **`USERS.TYPE` may not be populated in older accounts.** If it's blank across the board,
  Section 6's service-account detection falls back to naming-convention matching — treat that
  as a starting point to confirm with the team, not a definitive list.
- **Confirm before revoking anything.** Every result here is meant to narrow down what to look
  at and prioritize a conversation with the object/role owner — not to hand you a list to
  revoke unilaterally, especially at a regulated healthcare org where an access change can
  have compliance implications.
